In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D5 — IMF Country Focus — Booming India at risk of overheating
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip -q install pymupdf

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re

import fitz
import pandas as pd

DOCUMENT_ID = "D5"
DOCUMENT_NAME = "IMF Country Focus — Booming India at risk of overheating"
BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "Layout-aware PyMuPDF block conversion with deterministic "
    "statistical-table reconstruction"
)

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "ce59b2fb51cfcee5b579dd713c59ae5a5e14b57f66eab40d4cbaee151a62064d"
EXPECTED_PAGE_COUNT = 2

EXPECTED_RECORD_COUNT = 44
EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

EXPECTED_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

KNOWN_HEADINGS = [
    "Booming India at risk of overheating",
    "India at a glance",
    "Taking off",
    "Ease up on the monetary accelerator",
    "Reduce debt to finance development",
    "Develop broader and deeper capital markets",
    "Promote job growth and bolster the infrastructure",
    "Inflation risks"
]

EXPECTED_COMPONENTS = {
    "article_title": "Booming India at risk of overheating",
    "country_profile": "India at a glance",
    "chart": "Taking off",
    "monetary_section": "Ease up on the monetary accelerator",
    "fiscal_section": "Reduce debt to finance development",
    "capital_markets_section": "Develop broader and deeper capital markets",
    "employment_section": "Promote job growth and bolster the infrastructure",
    "statistical_table": "Inflation risks",
    "author": "Charles Kramer",
    "publication_date": "April 11, 2007"
}

OUTPUT_DIR = Path("outputs_D5_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Expected source pages:", EXPECTED_PAGE_COUNT)
print("Fixed Stage 1 reference records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ============================================================
# 1. Source document upload and identity verification
# ============================================================

uploaded = files.upload()

pdf_files = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError("Upload exactly one original D5 PDF.")

SOURCE_PATH = pdf_files[0]

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError("Unexpected source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded D5 PDF does not match the frozen source identity.")

document = fitz.open(SOURCE_PATH)

if len(document) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {len(document)}."
    )

if document.needs_pass:
    raise ValueError("The D5 source requires a password.")

page_texts = [
    page.get_text("text", sort=True)
    for page in document
]

full_text = "\n".join(page_texts)
text_layer_present = bool(full_text.strip())

component_checks = {
    key: marker in full_text
    for key, marker in EXPECTED_COMPONENTS.items()
}

if not text_layer_present:
    raise ValueError("D5 should contain a machine-readable text layer.")

if not all(component_checks.values()):
    raise ValueError(
        "One or more expected D5 source components are missing."
    )

SOURCE_CHECK = {
    "document_id": DOCUMENT_ID,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "page_count": len(document),
    "page_count_valid": len(document) == EXPECTED_PAGE_COUNT,
    "machine_readable_text_layer": text_layer_present,
    "expected_component_checks": component_checks,
    "all_expected_components_present": all(component_checks.values())
}

print(json.dumps(SOURCE_CHECK, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 2. Recover non-empty PDF text blocks
# ============================================================

def preserve_block_text(value):
    if value is None:
        return None

    text = str(value).replace("\r\n", "\n").replace("\r", "\n")

    # Representation-only whitespace cleanup.
    lines = [line.rstrip() for line in text.splitlines()]
    text = "\n".join(lines).strip()

    return text if text else None

block_records = []

for page_index, page in enumerate(document):
    page_number = page_index + 1

    for block_position, block in enumerate(
        page.get_text("blocks", sort=False)
    ):
        text = preserve_block_text(block[4])

        if not text:
            continue

        block_records.append({
            "Page Number": page_number,
            "Block Position": block_position,
            "Block Number": int(block[5]) if len(block) > 5 else block_position,
            "Block Type": int(block[6]) if len(block) > 6 else 0,
            "X0": float(block[0]),
            "Y0": float(block[1]),
            "X1": float(block[2]),
            "Y1": float(block[3]),
            "Text": text
        })

blocks_df = pd.DataFrame(block_records)

print("Retained source blocks:", len(blocks_df))
display(blocks_df.head())


In [ ]:
# ============================================================
# 3. Deterministic layout order and regions assignment
# ============================================================

def layout_region(page_number, x0, y0, page_width, page_height):
    rx = x0 / page_width
    ry = y0 / page_height

    if ry < 0.08:
        return "Page header"

    if rx < 0.50:
        return f"Page {page_number} left region"

    return f"Page {page_number} right region"

ordered_parts = []

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    page = document[page_number - 1]
    subset = blocks_df[
        blocks_df["Page Number"] == page_number
    ].copy()

    subset["Layout Region"] = subset.apply(
        lambda row: layout_region(
            page_number,
            row["X0"],
            row["Y0"],
            page.rect.width,
            page.rect.height
        ),
        axis=1
    )

    region_rank = {
        "Page header": 0,
        f"Page {page_number} left region": 1,
        f"Page {page_number} right region": 2
    }

    subset["_region_rank"] = subset["Layout Region"].map(region_rank).fillna(9)

    subset = subset.sort_values(
        ["_region_rank", "Y0", "X0", "Block Position"]
    )

    ordered_parts.append(subset)

ordered_blocks_df = pd.concat(
    ordered_parts,
    ignore_index=True
)

ordered_blocks_df["Structural Order"] = range(
    1, len(ordered_blocks_df) + 1
)

print("Structurally ordered blocks:", len(ordered_blocks_df))


In [ ]:
# ============================================================
# 4. Deterministic reconstruction of the Inflation risks table
# ============================================================

page2 = document[1]
page2_dict = page2.get_text("dict")

spans = []

for block in page2_dict["blocks"]:
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            text = span.get("text", "")
            if text.strip():
                spans.append({
                    "text": text.strip(),
                    "size": float(span["size"]),
                    "x0": float(span["bbox"][0]),
                    "y0": float(span["bbox"][1]),
                    "x1": float(span["bbox"][2]),
                    "y1": float(span["bbox"][3])
                })

span_df = pd.DataFrame(spans)


table_spans = span_df[
    (span_df["x0"] < 310)
    & (span_df["y0"] >= 600)
    & (span_df["y0"] <= 725)
].copy()

period_spans = table_spans[
    table_spans["text"].str.fullmatch(r"(?:19|20)\d{2}/\d{2}", na=False)
].sort_values("x0")

if len(period_spans) != 3:
    raise ValueError(
        f"Expected 3 table period headers; found {len(period_spans)}."
    )

period_headers = period_spans["text"].tolist()
period_x = period_spans["x0"].tolist()

status_spans = table_spans[
    table_spans["text"].isin(["Prov.", "Est."])
].copy()

column_status = {period: None for period in period_headers}

for _, status in status_spans.iterrows():
    nearest = min(
        range(len(period_x)),
        key=lambda i: abs(period_x[i] - status["x0"])
    )
    column_status[period_headers[nearest]] = status["text"]

display_period_headers = [
    (
        period
        if column_status[period] is None
        else f"{period} {column_status[period]}"
    )
    for period in period_headers
]

unit_markers = []

for y_group, group in table_spans.groupby(
    table_spans["y0"].round(0)
):
    joined = " ".join(group.sort_values("x0")["text"].tolist()).strip()

    if joined in {
        "(percent change)",
        "(percent of GDP)",
        "(million dollars)"
    }:
        unit_markers.append({
            "y": float(group["y0"].mean()),
            "unit": joined.strip("()")
        })

unit_markers = sorted(unit_markers, key=lambda x: x["y"])


data_rows = []

for y_group, group in table_spans.groupby(
    table_spans["y0"].round(0)
):
    group = group.sort_values("x0")

    normal_spans = group[group["size"] >= 6.0]

    numeric_candidates = normal_spans[
        normal_spans["text"].str.fullmatch(
            r"[–−-]?\d+(?:\.\d+)?",
            na=False
        )
        & (normal_spans["x0"] >= 170)
        & (normal_spans["x0"] <= 295)
    ]

    if len(numeric_candidates) != 3:
        continue

    values = numeric_candidates["text"].tolist()

    label_spans = normal_spans[
        normal_spans["x0"] < 170
    ]

    label = " ".join(label_spans["text"].tolist()).strip()

    if not label:
        continue

    row_y = float(group["y0"].mean())

    inline_unit_match = re.search(
        r"\(([^()]*(?:dollars|percent)[^()]*)\)",
        label,
        flags=re.IGNORECASE
    )

    if inline_unit_match:
        unit = inline_unit_match.group(1)
        label = re.sub(
            r"\s*\([^()]*\)\s*$",
            "",
            label
        ).strip()
    else:
        prior_markers = [
            marker
            for marker in unit_markers
            if marker["y"] < row_y
        ]
        unit = prior_markers[-1]["unit"] if prior_markers else None

    data_rows.append({
        "Indicator": label,
        "Unit": unit,
        display_period_headers[0]: values[0],
        display_period_headers[1]: values[1],
        display_period_headers[2]: values[2]
    })

table_df = pd.DataFrame(data_rows)

EXPECTED_TABLE_INDICATORS = [
    "Real GDP",
    "Wholesale prices",
    "General government debt",
    "Current account balance",
    "External debt",
    "Gross reserves"
]

if table_df["Indicator"].tolist() != EXPECTED_TABLE_INDICATORS:
    raise ValueError(
        "Deterministic table reconstruction did not recover the expected "
        "source row labels in order."
    )

footnote_spans = span_df[
    (span_df["x0"] < 310)
    & (span_df["y0"] >= 710)
    & (span_df["y0"] <= 725)
].sort_values(["y0", "x0"])

footnote_text = " ".join(footnote_spans["text"].tolist()).strip()

print("Reconstructed table:")
display(table_df)
print("Footnote:", footnote_text)


In [ ]:
# ============================================================
# 5. Complete structural Markdown
# ============================================================

def first_line_heading(text):
    lines = text.splitlines()
    if not lines:
        return None, text

    first = lines[0].strip()

    for heading in KNOWN_HEADINGS:
        if first.casefold() == heading.casefold():
            remainder = "\n".join(lines[1:]).strip()
            return heading, remainder

    return None, text

def escape_md_cell(value):
    if value is None:
        return ""
    return str(value).replace("|", "\\|").replace("\n", "<br>")

table_headers = table_df.columns.tolist()

table_md = [
    "| " + " | ".join(table_headers) + " |",
    "| " + " | ".join(["---"] * len(table_headers)) + " |"
]

for _, row in table_df.iterrows():
    table_md.append(
        "| "
        + " | ".join(escape_md_cell(row[col]) for col in table_headers)
        + " |"
    )

markdown_lines = [
    "# D5 — Booming India at risk of overheating",
    "",
    "> Complete structural conversion of the original two-page PDF.",
    "> All non-empty source text blocks are retained.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_lines += [
        f"## Source Page {page_number}",
        ""
    ]

    page_subset = ordered_blocks_df[
        ordered_blocks_df["Page Number"] == page_number
    ]

    for _, row in page_subset.iterrows():
        markdown_lines += [
            f"### Source Block {int(row['Structural Order'])}",
            f"- Layout Region: `{row['Layout Region']}`",
            (
                f"- Bounding Box: "
                f"`({row['X0']:.2f}, {row['Y0']:.2f}, "
                f"{row['X1']:.2f}, {row['Y1']:.2f})`"
            ),
            ""
        ]

        heading, remainder = first_line_heading(row["Text"])

        if heading is not None:
            markdown_lines += [
                f"#### {heading}",
                ""
            ]
            if remainder:
                markdown_lines += [
                    "```text",
                    remainder,
                    "```",
                    ""
                ]
        else:
            markdown_lines += [
                "```text",
                row["Text"],
                "```",
                ""
            ]

    if page_number == 2:
        markdown_lines += [
            "## Structural interpretation — Inflation risks table",
            "",
            *table_md,
            "",
            "### Table footnote",
            "",
            "```text",
            footnote_text,
            "```",
            ""
        ]

STRUCTURAL_MARKDOWN = "\n".join(markdown_lines).rstrip() + "\n"

REPRESENTATION_PATH = OUTPUT_DIR / "D5_branch_B_structural_markdown.md"
REPRESENTATION_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(REPRESENTATION_PATH)

print("Saved:", REPRESENTATION_PATH)
print("Representation SHA-256:", REPRESENTATION_SHA256)
print("Characters:", len(STRUCTURAL_MARKDOWN))


In [ ]:
# ============================================================
# 6. Conversion-integrity verification
# ============================================================

missing_block_content = []

for _, row in ordered_blocks_df.iterrows():
    text = row["Text"]
    heading, remainder = first_line_heading(text)

    if heading is None:
        if text not in STRUCTURAL_MARKDOWN:
            missing_block_content.append(
                int(row["Structural Order"])
            )
    else:
        if heading not in STRUCTURAL_MARKDOWN:
            missing_block_content.append(
                int(row["Structural Order"])
            )
        elif remainder and remainder not in STRUCTURAL_MARKDOWN:
            missing_block_content.append(
                int(row["Structural Order"])
            )

expected_table_values = {
    "Real GDP": ["7.5", "9.0", "8.9"],
    "Wholesale prices": ["6.5", "4.4", "6.4"],
    "General government debt": ["85.7", "81.9", "79.3"],
    "Current account balance": ["–2.5", "–9.1", "–22.7"],
    "External debt": ["17.7", "15.7", "18.1"],
    "Gross reserves": ["141.5", "151.6", "198.6"]
}

table_integrity_issues = []

for indicator, expected_values in expected_table_values.items():
    row = table_df[table_df["Indicator"] == indicator]

    if len(row) != 1:
        table_integrity_issues.append({
            "indicator": indicator,
            "issue": "row_count"
        })
        continue

    observed_values = row.iloc[0, 2:].tolist()

    if observed_values != expected_values:
        table_integrity_issues.append({
            "indicator": indicator,
            "expected_values": expected_values,
            "observed_values": observed_values
        })

page_markers_valid = all(
    f"## Source Page {p}" in STRUCTURAL_MARKDOWN
    for p in range(1, EXPECTED_PAGE_COUNT + 1)
)

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": len(document),
    "converted_page_count": EXPECTED_PAGE_COUNT,
    "conversion_method": CONVERSION_METHOD,
    "all_source_pages_retained": page_markers_valid,
    "source_block_count": int(len(blocks_df)),
    "retained_source_block_count": int(len(ordered_blocks_df)),
    "excluded_source_block_count": 0,
    "missing_retained_block_count": len(missing_block_content),
    "missing_retained_blocks": missing_block_content,
    "table_reconstruction_method":
        "Deterministic source-geometry reconstruction from PyMuPDF spans",
    "manual_table_fallback_used": False,
    "table_record_count": int(len(table_df)),
    "table_record_count_valid": len(table_df) == 6,
    "table_integrity_issue_count": len(table_integrity_issues),
    "table_integrity_issues": table_integrity_issues,
    "footnote_preserved": bool(footnote_text),
    "ocr_applied": False,
    "page_cropping_applied": False,
    "out_of_scope_content_removed": False,
    "chart_values_estimated": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed": all([
        SOURCE_HASH_MATCH,
        page_markers_valid,
        len(missing_block_content) == 0,
        len(table_df) == 6,
        len(table_integrity_issues) == 0,
        bool(footnote_text)
    ])
}

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D5_branch_B_conversion_integrity.json"
)

with open(CONVERSION_INTEGRITY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        CONVERSION_INTEGRITY,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(CONVERSION_INTEGRITY, indent=2, ensure_ascii=False))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError("D5 Branch B conversion integrity failed.")


In [ ]:
# ============================================================
# 7. Preservation of conversion audits and representation metadata
# ============================================================

BLOCK_AUDIT_PATH = OUTPUT_DIR / "D5_branch_B_block_audit.csv"
TABLE_AUDIT_PATH = OUTPUT_DIR / "D5_branch_B_table_audit.csv"

ordered_blocks_df.drop(
    columns=["_region_rank"],
    errors="ignore"
).to_csv(
    BLOCK_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

table_df.to_csv(
    TABLE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "conversion_method": CONVERSION_METHOD,
    "llm_input_representation": "Structural Markdown",
    "representation_type":
        "Complete PDF converted to layout-aware structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "complete_source_pdf_retained": True,
    "all_nonempty_source_text_blocks_retained": True,
    "out_of_scope_content_retained": True,
    "page_boundaries_made_explicit": True,
    "source_block_boundaries_made_explicit": True,
    "layout_regions_made_explicit": True,
    "known_headings_made_explicit": True,
    "statistical_table_structurally_reconstructed": True,
    "table_reconstruction_method":
        "Deterministic source-geometry reconstruction from PyMuPDF spans",
    "manual_table_fallback_used": False,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "chart_values_estimated": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D5_branch_B_representation.json"
)

with open(REPRESENTATION_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        REPRESENTATION_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    REPRESENTATION_METADATA,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 8. Definition of fixed Branch B extraction prompt
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "records": [
        {
            "Category": None,
            "Indicator or Policy Area": None,
            "Value": None,
            "Unit": None,
            "Qualifier": None,
            "Reference Period": None,
            "Description": None,
            "Source Location": None
        }
    ]
}

PROMPT_TEXT = f"""You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached structurally converted Markdown document
for the article “Booming India at risk of overheating.”

Treat the attached structurally converted Markdown document as the only
source of information.

Include:

1. Each principal policy measure introduced by the sentence
   “A combination of these four main policy measures is critical”.
2. Every item represented in the “India at a glance” country-profile
   region.
3. Every explicitly stated quantitative observation in the narrative
   article that belongs to the defined extraction scope.
4. The explicit quantitative statement represented in the “Taking off”
   chart caption.
5. Every numeric observation represented in the structurally
   reconstructed “Inflation risks” statistical table.

Exclude:

- values inferred or estimated from plotted chart lines;
- chart axis values;
- chart series dates and chart legend dates;
- page numbers and publication metadata;
- the photograph and photograph caption;
- promotional content;
- copyright text and publisher branding;
- values appearing only in source citations;
- qualitative statements that are not one of the principal policy
  measures;
- detailed policy sub-actions that merely elaborate a principal policy
  measure;
- Markdown representation metadata, source-block identifiers,
  coordinates and layout-region labels.

For every included record, extract:

- Category
- Indicator or Policy Area
- Value
- Unit
- Qualifier
- Reference Period
- Description
- Source Location

Category rules:

Use exactly one of:

- Main policy measure
- Country profile
- Narrative quantitative observation
- Statistical table observation

Indicator or Policy Area:

- Provide a concise source-grounded name for the represented policy
  area, profile item or quantitative indicator.
- Do not merge separate observations merely because they concern a
  similar topic.

Value:

- Preserve the explicitly represented source value.
- Use a JSON number for quantitative values.
- Use a JSON string for textual values.
- Keep negative numbers negative.
- Do not calculate, convert, reinterpret or estimate values.

Unit:

- Preserve the source-grounded measurement unit.
- Use "text" for textual policy or profile values.
- Do not silently correct source units.
- Preserve the unit printed for Gross reserves even if it appears
  unusual.

Qualifier:

- Preserve explicit approximation or inequality qualifiers such as
  "about", "around", "just over", "more than" and "nearly".
- Use null when no such qualifier is explicitly associated with the
  value.
- Do not use table-column status labels such as Prov. or Est. as
  Qualifier; those may be retained in Description where relevant.

Reference Period:

- Preserve explicitly associated fiscal years, calendar years,
  durations or relative periods.
- Use null when no explicit period is associated with the record.

Description:

- Provide a concise source-grounded explanation.
- Do not introduce external interpretation.

Source Location:

Use one of these exact labels when applicable:

- Page 1 — Four main policy measures
- Page 1 — India at a glance
- Page 1 — Opening narrative
- Page 1 — Taking off chart caption
- Page 1 — Ease up on the monetary accelerator
- Page 1 — Reduce debt to finance development
- Page 2 — Reduce debt to finance development
- Page 2 — Promote job growth and bolster the infrastructure
- Page 2 — Inflation risks table

Additional rules:

- Use the explicit structural cues in the Markdown while remaining
  grounded in source content.
- Distinguish narrative observations from statistical-table
  observations.
- Preserve decimal precision as represented.
- Preserve negative signs.
- Do not use external knowledge.
- Do not follow hyperlinks.
- Return one record for every included source observation.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.

Expected JSON schema:
{json.dumps(EXPECTED_OUTPUT_STRUCTURE, indent=2, ensure_ascii=False)}

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D5_branch_B_prompt.txt"
PROMPT_PATH.write_text(PROMPT_TEXT, encoding="utf-8")

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print(PROMPT_TEXT)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 9. Creation of Branch B experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_format": "PDF",
    "llm_input_representation":
        "Structural Markdown",
    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "complete_original_pdf_content_retained": True,
    "out_of_scope_content_retained": True,
    "layout_aware_block_extraction_applied": True,
    "page_boundaries_preserved": True,
    "layout_regions_made_explicit": True,
    "source_block_boundaries_made_explicit": True,
    "statistical_table_reconstructed": True,
    "table_reconstruction_method":
        "Deterministic source-geometry reconstruction from PyMuPDF spans",
    "manual_table_fallback_used": False,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "chart_line_values_estimated": False,
    "manual_correction_applied": False,
    "content_validation_performed": False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
}

METADATA_PATH = OUTPUT_DIR / "D5_branch_B_experiment_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        EXPERIMENT_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    EXPERIMENT_METADATA,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 10. Download files for independent LLM execution
# ============================================================

for path in [
    REPRESENTATION_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    BLOCK_AUDIT_PATH,
    TABLE_AUDIT_PATH,
    PROMPT_PATH,
    METADATA_PATH
]:
    files.download(path)

print(
    "\nIndependent execution instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D5_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D5_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF or Stage 1 reference values.\n"
    "5. Do not manually correct, regenerate or repair the response.\n"
    "6. Save the complete response exactly as returned in a TXT file."
)


In [ ]:
# ============================================================
# 11. Upload and preserve untouched Branch B response
# ============================================================

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete D5 Branch B response."
    )

RAW_SOURCE_PATH = Path(next(iter(uploaded_output)))

RAW_RESPONSE_TEXT = RAW_SOURCE_PATH.read_text(encoding="utf-8")

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError("The uploaded model response is empty.")

RAW_RESPONSE_PATH = OUTPUT_DIR / "D5_branch_B_raw_response.txt"
RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved:", RAW_RESPONSE_PATH)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 12. Parse without modifying or repairing the response
# ============================================================

valid_json = False
json_error = None
parsed_extraction = None

try:
    parsed_extraction = json.loads(RAW_RESPONSE_TEXT)
    valid_json = True
except json.JSONDecodeError as exc:
    json_error = str(exc)

print("Valid JSON:", valid_json)
print("JSON error:", json_error)


In [ ]:
# ============================================================
# 13. Validate top-level output structure
# ============================================================

top_level_object_valid = (
    valid_json
    and isinstance(parsed_extraction, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_extraction
)
document_id_correct = (
    document_id_present
    and parsed_extraction.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_extraction
)
branch_correct = (
    branch_present
    and parsed_extraction.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_extraction
)
records_is_list = (
    records_present
    and isinstance(parsed_extraction.get("records"), list)
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list
])

records = (
    parsed_extraction["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(records)
    if records_evaluable
    else None
)

record_count_valid = (
    observed_record_count == EXPECTED_RECORD_COUNT
    if records_evaluable
    else None
)

print("Records evaluable:", records_evaluable)
print("Observed records:", observed_record_count)
print("Record count matches Stage 1:", record_count_valid)


In [ ]:
# ============================================================
# 14. Record-schema and field-type checks
# ============================================================

record_structure_issues = []
field_type_issues = []

for index, record in enumerate(records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record)
    expected_fields = set(EXPECTED_FIELDS)

    missing = sorted(expected_fields - actual_fields)
    extra = sorted(actual_fields - expected_fields)

    if missing or extra:
        record_structure_issues.append({
            "record_index": index,
            "missing_fields": missing,
            "extra_fields": extra
        })

    for field in [
        "Category",
        "Indicator or Policy Area",
        "Unit",
        "Qualifier",
        "Reference Period",
        "Description",
        "Source Location"
    ]:
        value = record.get(field)
        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": index,
                "field": field,
                "observed_type": type(value).__name__
            })

    value = record.get("Value")
    if (
        isinstance(value, bool)
        or (
            value is not None
            and not isinstance(value, (str, int, float))
        )
    ):
        field_type_issues.append({
            "record_index": index,
            "field": "Value",
            "observed_type": type(value).__name__
        })

records_with_structure_issues = len(record_structure_issues)
records_with_type_issues = len({
    item["record_index"]
    for item in field_type_issues
})

print("Records with structure issues:", records_with_structure_issues)
print("Records with type issues:", records_with_type_issues)


In [ ]:
# ============================================================
# 15. Content/scope diagnostics kept separate from schema validity
# ============================================================

category_issues = []
observed_category_counts = None
category_counts_valid = None

if records_evaluable:
    for index, record in enumerate(records):
        if not isinstance(record, dict):
            continue

        if record.get("Category") not in ALLOWED_CATEGORIES:
            category_issues.append({
                "record_index": index,
                "observed_category": record.get("Category")
            })

    observed_category_counts = {
        category: sum(
            1
            for record in records
            if isinstance(record, dict)
            and record.get("Category") == category
        )
        for category in EXPECTED_CATEGORY_COUNTS
    }

    category_counts_valid = (
        observed_category_counts == EXPECTED_CATEGORY_COUNTS
    )

missing_values_by_field = (
    {
        field: sum(
            1
            for record in records
            if not isinstance(record, dict)
            or field not in record
            or record.get(field) is None
        )
        for field in EXPECTED_FIELDS
    }
    if records_evaluable
    else None
)

def record_key(record):
    return (
        record.get("Category"),
        record.get("Indicator or Policy Area"),
        record.get("Reference Period"),
        record.get("Source Location")
    )

duplicate_record_keys = []

if records_evaluable:
    keys = [record_key(record) for record in records if isinstance(record, dict)]
    duplicate_record_keys = sorted(
        {key for key in keys if keys.count(key) > 1},
        key=str
    )

# Table diagnostics
table_records = [
    record
    for record in records
    if isinstance(record, dict)
    and record.get("Category") == "Statistical table observation"
]

table_record_count_valid = (
    len(table_records) == 18
    if records_evaluable
    else None
)

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference": record_count_valid,
    "observed_category_counts": observed_category_counts,
    "category_counts_match_reference": category_counts_valid,
    "invalid_category_count":
        len(category_issues) if records_evaluable else None,
    "duplicate_record_key_count":
        len(duplicate_record_keys) if records_evaluable else None,
    "table_record_count":
        len(table_records) if records_evaluable else None,
    "table_record_count_valid": table_record_count_valid,
    "missing_values_by_field": missing_values_by_field
}

print(json.dumps(
    CONTENT_DIAGNOSTICS,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 16. Structural/schema validity
# ============================================================

record_schema_valid = (
    records_with_structure_issues == 0
    if records_evaluable
    else False
)

field_types_valid = (
    records_with_type_issues == 0
    if records_evaluable
    else False
)

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid,
    field_types_valid
])

TECHNICAL_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "valid_json": bool(valid_json),
    "json_parsing_error": json_error,

    "top_level_object_valid":
        bool(top_level_object_valid),
    "document_id_present":
        bool(document_id_present),
    "document_id_correct":
        bool(document_id_correct),
    "branch_present":
        bool(branch_present),
    "branch_correct":
        bool(branch_correct),
    "records_present":
        bool(records_present),
    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        observed_record_count,
    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_valid":
        category_counts_valid,

    "records_with_structure_issues":
        records_with_structure_issues
        if records_evaluable else None,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "records_with_type_issues":
        records_with_type_issues
        if records_evaluable else None,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "record_schema_valid":
        bool(record_schema_valid),

    "field_types_valid":
        bool(field_types_valid),

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D5_branch_B_technical_diagnostics.json"
)

with open(TECHNICAL_DIAGNOSTICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        TECHNICAL_DIAGNOSTICS,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    TECHNICAL_DIAGNOSTICS,
    indent=2,
    ensure_ascii=False
))



In [ ]:
# ============================================================
# 17. Parsed extraction only if structurally evaluable
# ============================================================

PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D5_branch_B_parsed_extraction.json"

if structurally_evaluable:
    with open(PARSED_EXTRACTION_PATH, "w", encoding="utf-8") as f:
        json.dump(
            parsed_extraction,
            f,
            indent=2,
            ensure_ascii=False
        )

    PARSED_EXTRACTION_SHA256 = sha256_file(PARSED_EXTRACTION_PATH)
    print("Parsed extraction saved:", PARSED_EXTRACTION_PATH)
else:
    PARSED_EXTRACTION_SHA256 = None
    print("No parsed extraction created.")


In [ ]:
# =======================================================
# 18. Branch B experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        "Structural Markdown",

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "content_validation_performed":
        False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_source_content_retained": True,
    "normalisation_applied": False,
    "manual_table_fallback_used": False,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "valid_json": bool(valid_json),
    "records_evaluable": bool(records_evaluable),
    "structurally_evaluable": bool(structurally_evaluable),
    "record_count_matches": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_match": category_counts_valid,
    "records_with_structure_issues":
        records_with_structure_issues if records_evaluable else None,
    "records_with_type_issues":
        records_with_type_issues if records_evaluable else None,
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_created":
        bool(structurally_evaluable),
    "parsed_extraction_sha256": PARSED_EXTRACTION_SHA256,
    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D5."
    )
}

SUMMARY_PATH = OUTPUT_DIR / "D5_branch_B_experiment_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        EXPERIMENT_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    EXPERIMENT_SUMMARY,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 19. Final artefact inventory and downloads
# ============================================================

generated_outputs = [
    REPRESENTATION_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    BLOCK_AUDIT_PATH,
    TABLE_AUDIT_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SUMMARY_PATH
]

if structurally_evaluable:
    generated_outputs.append(
        PARSED_EXTRACTION_PATH
    )
print("Generated D5 Branch B outputs:")

for path in generated_outputs:
    print("-", path.name)

for path in generated_outputs:
    files.download(path)
